In [89]:
import numpy as np
import pandas as pd
import string
import re
from collections import Counter

In [90]:
df = pd.read_csv("Turkish_Book_Dataset_Kaggle_V2.csv", dtype=str)

In [91]:
df.head()

,book_title,book_publisher,book_author,book_category_name,book_productcode,book_page_count,book_released_year,book_detail
0,Klinik Uygulamada Bilişsel Davranışçı Terapi - 2,Nobel Akademik Yayıncılık,Mehmet Ak,academic,9786254272936,260,2022,Bilişsel davranışçı terapi (BDT) alanında çalı...
1,Taşra Üniversiteleri,İletişim Yayınevi,Tuğba Tekerek,academic,9789750534218,400,2023,"Tugˆba Tekerek, tas¸ra u¨niversitelerinin akad..."
2,Çocuklar ve Ergenler İçin Bilişsel Davranışçı ...,Nobel Akademik Yayıncılık,Arzu Aydın,academic,9786050330076,304,2021,Çocuklar ve Ergenler İçin Bilişsel-davranışçı ...
3,İki Dirhem Bir Çekirdek,Kapı Yayınları,İskender Pala,academic,9789758950379,212,2022,"Anlatımı güzelleştirmek, savunulan fikir ve dü..."
4,Reader at Work 2,ODTÜ Geliştirme Vakfı Yayıncılık,Kolektif,academic,9789754293999,550,2020,This book is the second volume of a collection...


In [92]:
df.book_author.value_counts()[:10]

book_author
Kolektif                         2773
Sigmund Freud                     167
Tahir Özakkaş                     159
Stefan Zweig                      126
Lev Nikolayeviç Tolstoy           118
Prof. Dr. Bahri Öztürk            117
Jack London                       117
Hüseyin Rahmi Gürpınar            114
Sabahattin Ali                    114
Fyodor Mihayloviç Dostoyevski     108
Name: count, dtype: int64

In [93]:
yazar_0 = "Sabahattin Ali"
yazar_1 = "Stefan Zweig"

In [94]:
# FARK: dosyada boş satırlar, çok kısa yazılar ve tekrarlar var, eliyoruz
df = df.dropna(subset=["book_detail", "book_author"])
df = df[df.book_detail.str.len() >= 50]
df = df.drop_duplicates("book_detail")

In [95]:
kitaplar_0 = df[df["book_author"] == yazar_0]
kitaplar_1 = df[df.book_author == yazar_1]

In [96]:
print(len(kitaplar_0), len(kitaplar_1))

109 100


In [97]:
kitaplar_0.book_author.unique()

<StringArray>
['Sabahattin Ali']
Length: 1, dtype: str

In [98]:
kitaplar_1.book_author.unique()

<StringArray>
['Stefan Zweig']
Length: 1, dtype: str

In [99]:
kitaplar_0 = kitaplar_0.book_detail.to_numpy()
kitaplar_1 = kitaplar_1.book_detail.to_numpy()

In [100]:
print(yazar_0, "ilk kitabi:\n\n")
print(kitaplar_0[0])

Sabahattin Ali ilk kitabi:


”Dünya’nın en basit, en zavallı, hatta en ahmak adamı bile, insanı hayretten hayrete düşürecek ne müthiş ve karışık bir ruha maliktir!... Niçin bunu anlamaktan bu kadar kaçıyor ve insan dedikleri mahluku anlaşılması ve hakkında hüküm verilmesi en kolay şeylerden biri zannediyoruz?”


In [101]:
print(yazar_1, "ilk kitabi:\n\n")
print(kitaplar_1[0])

Stefan Zweig ilk kitabi:


… Sözlerin, konuşulanların nasıl bir mucize yaratacağını, dudağın boşluğa doğru yaptığı büyülü titreşimin sayısız dünyalar kurup sayısız dünyaları yıktığını kim bilir?Modern bilimsel tıp, hastalığı insanın bütününe değil, organlarından birine ilişkin bir şeymiş gibi algılar. Oysa *hekimlik sanatı*nın mucizeleri, bedenle ruh arasındaki yüksek ilişkilere dayanır. Ruh Yoluyla Tedavi’nin sırrı, insanın varlığının bütününü hastalığın bütününe karşı harekete geçirmesi çağrısında gizlidir.Zweig, münferit tedavi ustaları olarak adlandırdığı Franz Anton Mesmer, Mary Baker-Eddy ve Sigmund Freud’un akademik tıptan ayrılarak yaptıkları başkaldırıyı ve kitleleri peşlerinden sürükleyerek elde ettikleri dramatik başarıyı kültür tarihinin en ilginç olaylarından biri olarak görüyor. Bu kişiler farklı, hatta birbirine zıt yollardan giderek ruh yoluyla tedavinin aynı ilkesini yüz binlerce kişide gerçekleştirmişlerdir.Zweig, bu kitapta üç mucizevi tedavi yönteminin kahramanı ola

In [102]:
kitaplar = [kitaplar_0, kitaplar_1]
yazarlar = [yazar_0, yazar_1]

In [103]:
satirlar = []
labels = []

In [104]:
# FARK: tanitim yazilarinda satir yok, bu yuzden cumlelere boluyoruz
# FARK: yazarin adini metinden siliyoruz (yoksa model adi okuyarak hile yapar)
tablo = str.maketrans("", "", string.punctuation + "…“”‘’")

def kucult(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()

def temizle(satir, yazar):
    satir = kucult(satir).translate(tablo)
    isim_parcalari = [p for p in kucult(yazar).translate(tablo).split() if len(p) >= 3]
    return " ".join(k for k in satir.split() if not any(k.startswith(p) for p in isim_parcalari))

# kitap_seti: bir yazara ait tum kitaplar
for i, kitap_seti in enumerate(kitaplar):
    for kitap in kitap_seti:  # kitap: bir yazara ait kitap setindeki her bir kitabin tanitim yazisi
        for satir in re.split(r"(?<=[.!?])\s+|\n", kitap):
            # satiri "Hey! Naber?"den "hey naber"e donusturuyoruz
            satir = temizle(satir, yazarlar[i])
            if satir:
                satirlar.append(satir)
                labels.append(i)

In [105]:
satirlar[:5]

['dünyanın en basit en zavallı hatta en ahmak adamı bile insanı hayretten hayrete düşürecek ne müthiş ve karışık bir ruha maliktir',
 'niçin bunu anlamaktan bu kadar kaçıyor ve insan dedikleri mahluku anlaşılması ve hakkında hüküm verilmesi en kolay şeylerden biri zannediyoruz',
 'sabahatttin 1935 tarihli ilk öykü kitabı',
 'kitabın ismi aynı ada sahip öyküden gelmektedir',
 '3 kısımdan oluşan kitapta toplamda 16 adet öykü bulunmaktadır']

In [106]:
labels[:5]

[0, 0, 0, 0, 0]

In [107]:
satirlar[-5:]

['sevdiği adam dışında kimsenin varlığını önemsemeyen aşkın tek taraflı olup olamayacağını bir aşk için ne kadar fedakârlık yapılabileceğini aşkın tutkuya dönüşümünü ve saplantının insanı getirebileceği noktaları anlatan bu kadını dinlemeye hazır mısınızbilinmeyen bir kadının mektubusaplantılı bir aşka düşen bir kadının son ana kadar yaşadıklarını anlattığı göndereni belli olmayan bir itiraftır',
 'ünlü romancı r bir sabah beklemediği bir mektup alır',
 'içinden bir kadının tutkulu ve bir o kadar da saplantılı aşkı çıkar',
 'çocuk yaşından beri aklından bir an bile çıkarmamış iz bırakabilmek için tüm fırsatları değerlendirmiş ondan bir hatıra koparabilmek için geleceğini yok etmeye razı bir kadının kaleminden okudukları bu ünlü romancıyı hayrete düşürür',
 'sevdiği adam dışında kimsenin varlığını önemsemeyen aşkın tek taraflı olup olamayacağını bir aşk için ne kadar fedakârlık yapılabileceğini aşkın tutkuya dönüşümünü ve saplantının insanı getirebileceği noktaları anlatan bu kadını din

In [108]:
labels[-5:]

[1, 1, 1, 1, 1]

In [109]:
print(len(satirlar), len(labels))

1829 1829


In [110]:
from sklearn.model_selection import train_test_split

In [111]:
X_train, X_test, y_train, y_test = train_test_split(
    satirlar, labels, test_size=0.2, random_state=42)

In [112]:
print(len(X_train), len(y_train), len(X_test), len(y_test))

1463 1463 366 366


In [113]:
for i in range(15):
    print(X_train[i], "::", y_train[i], "::", yazarlar[y_train[i]])

halbuki ne şeytanı azizim ne şeytanı :: 0 :: Sabahattin Ali
önce ingiltereye 1940ta da brezilyaya göç etti :: 1 :: Stefan Zweig
başyapıtı sayılan satranç dünya edebiyatının klasikleri arasındadır :: 1 :: Stefan Zweig
dünya savaşı sırasında isviçrede ikamet etti :: 1 :: Stefan Zweig
nihayetinde kürk mantolu madonna olarak bugüne ulaşan eseri askerde hatta çatlak olan kolu sargıdayken yazdığı bilinmektedir :: 0 :: Sabahattin Ali
kuyucaklı yusufun önemi yalnızca başarılı bir roman olmasından ileri gelmez öncü bir yapıt olması da ona tarihsel açıdan bir önem kazandırır çünkü bu yapıt daha önceki türk romanından iki bakımdan ayrılır ve yeni bir yol açar :: 0 :: Sabahattin Ali
kuyucaklı yusufun önemi yalnızca başarılı bir roman olmasından ileri gelmez öncü bir yapıt olması da ona tarihsel açıdan bir önem kazandırır çünkü bu yapıt daha önceki türk romanından iki bakımdan ayrılır ve yeni bir yol açar :: 0 :: Sabahattin Ali
sonra gözlerini kapattı :: 1 :: Stefan Zweig
kültürel yönden de kendisi

In [114]:
for i in range(15):
    print(X_test[i], "::", y_test[i], "::", yazarlar[y_test[i]])

çıktıktan sonra ankarada almanca öğretmenliği ankara devlet konservatuvarında çevirmenlik öğretmenlik ve dramaturgluk yaptı :: 0 :: Sabahattin Ali
bir kadının hayatından 24 saat kitabının fiyatı nedir ve nasıl belirlenir :: 1 :: Stefan Zweig
hepsine uzun geceden sonra gelen tanın kızıllığını görmek nasip olsun :: 1 :: Stefan Zweig
1912de de new york kanada panama küba ve porto rikoyu kapsayan bir amerika turu yapmıştır :: 1 :: Stefan Zweig
richard bach martı jonathan livingston :: 1 :: Stefan Zweig
ruhen yalnız :: 0 :: Sabahattin Ali
4757 yaş arası :: 1 :: Stefan Zweig
yapıtlarında psikolojiyi odak noktası olarak belirleyen özellikle toplum dışında yaşayan insanların hikâyesini anlatmaya değer bulur :: 1 :: Stefan Zweig
burada ağabeyimin ve bütün hanedanımız efradının birer birer ve hicaplarından öldüklerini gördüm :: 0 :: Sabahattin Ali
o esrarlı bir dil konuşan ellerini sazın üzerinde hareket ettirmeye başlamış ve gözlerini yere yahut kucağından fırlamak ister gibi sıçrayan sazına di

In [115]:
w2i = {}  # kelime dağarcığı (vocab.) oluşturuyoruz
w2i["<UNK>"] = 0

In [116]:
# FARK: geçiş tablosu kelime x kelime büyüklüğünde, bellek için sadece en sık geçen 3000 kelimeyi alıyoruz
sayac = Counter(token for satir in X_train for token in satir.split())

index = 1
for token, adet in sayac.most_common(3000):
    w2i[token] = index
    index += 1

In [117]:
list(w2i.items())[:5]

[('<UNK>', 0), ('bir', 1), ('ve', 2), ('bu', 3), ('da', 4)]

In [118]:
len(w2i)

3001

In [119]:
print(w2i["bir"], w2i["ve"], w2i["bu"])

1 2 3


In [120]:
# "bir satranç ustası" satirini [14, 15, 16] listesine dönüştüreceğiz
X_train_int = []
X_test_int = []

In [121]:
X_train[:3]

['halbuki ne şeytanı azizim ne şeytanı',
 'önce ingiltereye 1940ta da brezilyaya göç etti',
 'başyapıtı sayılan satranç dünya edebiyatının klasikleri arasındadır']

In [122]:
# train setindeki text satirlari, int satirlara donusturuyoruz
for satir in X_train:
    satir_int = []
    tokens = satir.split()
    for token in tokens:
        # FARK: sozluk 3000 kelimeyle sinirli oldugu icin, train'de de sozlukte olmayan token'a <UNK> (0) veriyoruz
        satir_int.append(w2i.get(token, 0))

    X_train_int.append(satir_int)

In [123]:
X_train_int[:3]

[[752, 22, 484, 1046, 22, 484],
 [72, 1657, 1658, 4, 304, 2998, 149],
 [2999, 3000, 17, 28, 188, 0, 1659]]

In [124]:
# test setindeki satirlari, int satirlara donusturuyoruz
for satir in X_test:
    satir_int = []
    tokens = satir.split()
    for token in tokens:
        # eğer test setindeki bir token, train setinin sozlugunde w2i bulunmuyorsa
        # bu token için <UNK> tag'ine karsilik gelen 0 degerini atiyoruz
        satir_int.append(w2i.get(token, 0))

    X_test_int.append(satir_int)

In [125]:
X_test_int[:3]

[[2101, 19, 0, 818, 0, 0, 0, 0, 0, 0, 2, 0, 1025],
 [1, 12, 58, 37, 38, 48, 2234, 395, 2, 93, 2851],
 [0, 49, 0, 19, 171, 0, 0, 884, 0, 540]]

In [126]:
X_train_int_SA = []  # sadece Sabahattin Ali'ye ait satirlar
for i, label in enumerate(y_train):
    if label == 0:  # 0 == Sabahattin Ali
        X_train_int_SA.append(X_train_int[i])

In [127]:
len(X_train_int_SA)

514

In [128]:
y_train_0s = [l for l in y_train if l == 0]

In [129]:
len(y_train_0s)

514

In [130]:
X_train_int_SA[:3]

[[752, 22, 484, 1046, 22, 484],
 [0, 43, 39, 73, 16, 1047, 753, 106, 0, 261, 1662, 9, 485, 0, 88, 0],
 [53,
  402,
  1663,
  486,
  262,
  1,
  54,
  1664,
  754,
  1665,
  577,
  1,
  578,
  211,
  4,
  89,
  1048,
  350,
  1,
  1049,
  1666,
  263,
  3,
  578,
  18,
  1050,
  114,
  1667,
  33,
  579,
  1051,
  2,
  35,
  1,
  487,
  1052]]

In [131]:
X_train_int_SZ = []  # Sadece Stefan Zweig'a ait satirlar
for i, label in enumerate(y_train):
    if label == 1:  # 1 == Stefan Zweig
        X_train_int_SZ.append(X_train_int[i])

print(len(X_train_int_SZ))

949


In [132]:
M = len(w2i)  # dagarcik boyutu (vocab. size)
print(M)

# Sabahattin Ali modeli
pi_0 = np.ones(M)
A_0 = np.ones((M, M))

# Stefan Zweig modeli
pi_1 = np.ones(M)
A_1 = np.ones((M, M))

3001


In [133]:
def train_markov_model(satirlar_int, pi, A):
    for satir_int in satirlar_int:  # satir_int: [5928, 4336, 1535, 4397, 802]
        for i, token_index in enumerate(satir_int):  # i: 0, index: 5928
            if i == 0:
                # satir basindaysak, token'i pi'ye ekle
                pi[token_index] += 1
            else:
                # satir basinda degilsek,
                # bir onceki token'dan simdiki token'a gecis frekansini 1 arttir
                A[satir_int[i - 1], token_index] += 1

In [134]:
train_markov_model(X_train_int_SA, pi_0, A_0)

In [135]:
train_markov_model(X_train_int_SZ, pi_1, A_1)

In [136]:
pi_0[:5]

array([98., 14.,  4., 17.,  1.])

In [137]:
A_1[:5, :15]

array([[959., 108., 126.,  29.,  12.,  22.,  21.,   5.,  10.,  19.,   4.,
         12.,   5.,   5.,  18.],
       [146.,   1.,   1.,   1.,   1.,   1.,   1.,   1.,   1.,   2.,   1.,
          1.,  69.,   1.,   1.],
       [104.,   7.,   1.,  12.,   1.,   2.,   1.,   1.,   4.,   1.,   1.,
          1.,   1.,   3.,   1.],
       [ 49.,   2.,   1.,   1.,   2.,   1.,   1.,   1.,   1.,   1.,   1.,
          1.,   3.,   1.,   1.],
       [ 16.,   2.,   1.,   6.,   1.,   1.,   1.,   1.,   2.,   1.,   1.,
          2.,   1.,   3.,   1.]])

In [138]:
# normalizasyon
pi_0 = pi_0 / pi_0.sum()
pi_1 = pi_1 / pi_1.sum()

In [139]:
pi_1[:5]

array([0.03772152, 0.0121519 , 0.00177215, 0.01189873, 0.00025316])

In [140]:
test = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

test2 = test.sum(axis=0)

test3 = test.sum(axis=1)

test4 = test.sum(axis=1, keepdims=True)

print("original")
print(test)

print("-" * 20)
print("sum(axis=0)")
print(test2)

print("-" * 20)
print("sum(axis=1)")
print(test3)

print("-" * 20)
print("sum(axis=1, keepdims=True)")
print(test4)

original
[[1 2 3]
 [4 5 6]]
--------------------
sum(axis=0)
[5 7 9]
--------------------
sum(axis=1)
[ 6 15]
--------------------
sum(axis=1, keepdims=True)
[[ 6]
 [15]]


In [141]:
A_0 = A_0 / A_0.sum(axis=1, keepdims=True)
A_1 = A_1 / A_1.sum(axis=1, keepdims=True)

In [142]:
log_pi_0 = np.log(pi_0)
log_A_0 = np.log(A_0)

log_pi_1 = np.log(pi_1)
log_A_1 = np.log(A_1)

In [143]:
log_pi_1[:5]

array([-3.27752455, -4.41026985, -6.33556071, -4.43132326, -8.28147086])

In [144]:
log_A_1[:5, :4]

array([[-1.7924544 , -3.97621424, -3.82206356, -5.29104964],
       [-3.19419006, -8.17779668, -8.17779668, -8.17779668],
       [-3.48330895, -6.1817897 , -8.12769985, -5.6427932 ],
       [-4.17626933, -7.37494245, -8.06808963, -8.06808963],
       [-5.26366122, -7.34310276, -8.03624994, -6.24449047]])

In [145]:
count_0 = sum(y == 0 for y in y_train)  # SA'ya ait satirlarin sayisi
count_1 = sum(y == 1 for y in y_train)  # SZ'ya ait satirlarin sayisi

total = len(y_train)  # train setindeki toplam satir sayisi

# prior'lari hesapla
p_0 = count_0 / total
log_p_0 = np.log(p_0)

p_1 = count_1 / total
log_p_1 = np.log(p_1)

In [146]:
print(p_0, p_1, log_p_0, log_p_1)

0.3513328776486671 0.6486671223513328 -1.0460211355649593 -0.43283560241019653


In [147]:
def compute_log_prob(input, clas):
    """ 
    input'un verilen class'a ait olma olasiliginin log degerini hesaplar

    örn input: [1, 3, 2, 7, ...] 
    örn clas: 0 veya 1
    """
    pi = log_pi_0
    A = log_A_0
    prior = log_p_0

    if clas == 1:
        pi = log_pi_1
        A = log_A_1
        prior = log_p_1

    log_prob = 0
    for i, word_index in enumerate(input):
        if i == 0:
            log_prob += pi[word_index]
        else:
            log_prob += A[input[i - 1], word_index]

    log_prob += prior
    return log_prob

In [148]:
print(X_train_int[0], y_train[0])

[752, 22, 484, 1046, 22, 484] 0


In [149]:
compute_log_prob(X_train_int[0], 0)

np.float64(-39.83087372939613)

In [150]:
compute_log_prob(X_train_int[0], 1)

np.float64(-48.07092772374944)

In [151]:
def predict(inputs):
    predictions = []
    for input in inputs:
        probas = [compute_log_prob(input, 0), compute_log_prob(input, 1)]
        prediction = np.argmax(probas)
        predictions.append(prediction)

    return predictions

In [152]:
predictions_train = predict(X_train_int)
predictions_test = predict(X_test_int)

In [153]:
predictions_train[:5]

[np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0)]

In [154]:
y_train[:5]

[0, 1, 1, 1, 0]

In [155]:
predictions_train = np.array(predictions_train)
predictions_test = np.array(predictions_test)

In [156]:
predictions_train == y_train

array([ True,  True,  True, ...,  True,  True,  True], shape=(1463,))

In [157]:
train_accu = np.mean(predictions_train == y_train)
test_accu = np.mean(predictions_test == y_test)

In [158]:
print("Train accuracy:", train_accu)
print("Test accuracy:", test_accu)

Train accuracy: 0.9719753930280246
Test accuracy: 0.7896174863387978


In [159]:
import numpy as np 
import pandas as pd 
import string
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.naive_bayes import MultinomialNB

In [160]:
df.book_author.value_counts()

book_author
Kolektif           2371
Sigmund Freud       157
Tahir Özakkaş       146
Sabahattin Ali      109
Stefan Zweig        100
                   ... 
Simge Akpolat         1
Bahar Umurtak         1
Buğra Hararoğlu       1
Esra Arslan           1
Sevinç Yaman          1
Name: count, Length: 36491, dtype: int64

In [161]:
df_subset = df[df.book_author.isin(yazarlar)].copy()

In [162]:
df_subset.book_author.value_counts()

book_author
Sabahattin Ali    109
Stefan Zweig      100
Name: count, dtype: int64

In [163]:
df_subset.head()

,book_title,book_publisher,book_author,book_category_name,book_productcode,book_page_count,book_released_year,book_detail
13872,Ruh Yoluyla Tedavi,İmge Kitabevi Yayınları,Stefan Zweig,academic,9789755334059,364,2018,"… Sözlerin, konuşulanların nasıl bir mucize ya..."
34492,Freud: Mutluluğun Mimarı,Zeplin Kitap,Stefan Zweig,psychology,9786052060025,180,2017,"Zweig, Mutluluğun Mimarı’nda, Freud’un öğrenci..."
34499,Freud ve Öğretisi,Köprü Kitapları,Stefan Zweig,psychology,9786056644696,112,2017,"Stefan Zweig... Dünyanın en karanlık, en vahşi..."
36115,Bilinmeyen Bir Kadının Mektubu,İş Bankası Kültür Yayınları,Stefan Zweig,literature,9786254290350,56,2022,Stefan Zweig Bilinmeyen Bir Kadının Mektubu (B...
36878,Clarıssa,Palet Yayınları,Stefan Zweig,literature,9786258414806,152,2022,"Clarissa, Zweig’in ana kahramanının adını taşı..."


In [164]:
tablo = str.maketrans("", "", string.punctuation + "…“”‘’")

def kucult(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()

def clean(row):
    satirlar = row["book_detail"].split("\n")
    metin = kucult(" ".join(satirlar)).translate(tablo)
    # yazarın adının parçalarını (ve zweigın, alinin gibi eklerini) sil
    isim_parcalari = [p for p in kucult(row["book_author"]).translate(tablo).split() if len(p) >= 3]
    kelimeler = [k for k in metin.split() if not any(k.startswith(p) for p in isim_parcalari)]
    return " ".join(kelimeler)

df_subset["clean"] = df_subset.apply(lambda row: clean(row), axis = 1)

In [165]:
def etiketle(row):
    if row["book_author"] == yazarlar[0]:
        return 0 
    return 1

df_subset["labels"] = df_subset.apply(lambda row:  etiketle(row), axis=1)

In [166]:
df_subset.head()

,book_title,book_publisher,book_author,book_category_name,book_productcode,book_page_count,book_released_year,book_detail,clean,labels
13872,Ruh Yoluyla Tedavi,İmge Kitabevi Yayınları,Stefan Zweig,academic,9789755334059,364,2018,"… Sözlerin, konuşulanların nasıl bir mucize ya...",sözlerin konuşulanların nasıl bir mucize yarat...,1
34492,Freud: Mutluluğun Mimarı,Zeplin Kitap,Stefan Zweig,psychology,9786052060025,180,2017,"Zweig, Mutluluğun Mimarı’nda, Freud’un öğrenci...",mutluluğun mimarında freudun öğrencilik yıllar...,1
34499,Freud ve Öğretisi,Köprü Kitapları,Stefan Zweig,psychology,9786056644696,112,2017,"Stefan Zweig... Dünyanın en karanlık, en vahşi...",dünyanın en karanlık en vahşi bütün zamanları ...,1
36115,Bilinmeyen Bir Kadının Mektubu,İş Bankası Kültür Yayınları,Stefan Zweig,literature,9786254290350,56,2022,Stefan Zweig Bilinmeyen Bir Kadının Mektubu (B...,bilinmeyen bir kadının mektubu brief einer unb...,1
36878,Clarıssa,Palet Yayınları,Stefan Zweig,literature,9786258414806,152,2022,"Clarissa, Zweig’in ana kahramanının adını taşı...",clarissa ana kahramanının adını taşıyan kasım ...,1


In [167]:
df_subset.tail()

,book_title,book_publisher,book_author,book_category_name,book_productcode,book_page_count,book_released_year,book_detail,clean,labels
66144,Kürk Mantolu Madonna,Mahzen Yayıncılık,Sabahattin Ali,literature,9786056910647,192,2019,Koltuğundan kalkmadan elini uzattı. Sokuldum v...,koltuğundan kalkmadan elini uzattı sokuldum ve...,0
67003,Kendi Hayatının Şiirini Yazanlar,İş Bankası Kültür Yayınları,Stefan Zweig,literature,9786053604594,407,2012,"Stefan Zweig, Kendi Hayatının Şiirini Yazanlar...",kendi hayatının şiirini yazanlarda pek çok açı...,1
67363,Bütün Şiirleri,Mahzen Yayıncılık,Sabahattin Ali,literature,9786058085022,144,2019,"Göklerde kartal gibiydim, Kanatlarımdan vuruld...",göklerde kartal gibiydim kanatlarımdan vuruldu...,0
67389,Esirler,Mahzen Yayıncılık,Sabahattin Ali,literature,9786058085015,80,2019,"KÜRŞAD:Dostum, gözlerinde büyük endişelerin iz...",kürşaddostum gözlerinde büyük endişelerin izin...,0
67881,Bilinmeyen Bir Kadının Mektubu,Filozof Yayıncılık,Stefan Zweig,literature,9786050616989,55,2020,"Bilinmeyen Bir Kadının Mektubu,saplantılı bir ...",bilinmeyen bir kadının mektubusaplantılı bir a...,1


In [168]:
X = df_subset.clean.to_numpy()
y = df_subset.labels.to_numpy()

In [169]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [170]:
print(len(X_train), len(X_test))

167 42


In [171]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [172]:
len(vectorizer.vocabulary_)

7994

In [173]:
list(vectorizer.vocabulary_.items())[:10]

[('göklerde', 2799),
 ('kartal', 3892),
 ('gibiydim', 2722),
 ('kanatlarımdan', 3794),
 ('vuruldum', 6734),
 ('mor', 4809),
 ('çiçekli', 7576),
 ('dal', 1496),
 ('bahar', 802),
 ('vaktinde', 6615)]

In [174]:
vectorizer.get_feature_names_out()[:15]

array(['10', '100', '100üncü', '114', '120', '1200000den', '13', '134',
       '136', '14', '1424', '15', '16', '1622', '164'], dtype=object)

In [175]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17220 stored elements and shape (167, 7994)>

In [176]:
model = MultinomialNB()
model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[85.,82.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.68,-0.71]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 7994)","[[0.,3.,0.,...,1.,0.,0.], [7.,2.,1.,...,0.,1.,1.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 7994)","[[-9.68,-8.29,-9.68,...,-8.99,-9.68,-9.68], [-7.91,-8.89,-9.3 ,...,-9.99,-9.3 ,-9.3 ]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7994


In [177]:
model.score(X_train, y_train)

1.0

In [178]:
model.score(X_test, y_test)

0.9761904761904762

In [179]:
predictions_train = model.predict(X_train)
print("Train F1:", f1_score(y_train, predictions_train))

predictions_test = model.predict(X_test)
print("Test F1:", f1_score(y_test, predictions_test))

Train F1: 1.0
Test F1: 0.9714285714285714


In [180]:
y_train[:15]

array([0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1])

In [181]:
predictions_train[:15]

array([0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1])

In [182]:
sabahattin_ali_gibi = "Berlin'de yalnız yaşayan genç bir adamın, Ankara'da bir mantolu madonna tablosunun ardındaki hikayesi. Köy hayatı, yoksulluk ve aşk."

In [183]:
sa_vec = vectorizer.transform([sabahattin_ali_gibi])

In [184]:
model.predict(sa_vec)

array([0])

In [185]:
zweig_gibi = "Bir satranç ustasının zihnini ele geçiren saplantı ve Viyana'dan sürgüne giden bir adamın dünyası. Tek bir günde yaşanan tutku."
z_vec = vectorizer.transform([zweig_gibi])

In [186]:
model.predict(z_vec)

array([1])

In [187]:
df_subset[df_subset.clean.str.contains("satranç")]

,book_title,book_publisher,book_author,book_category_name,book_productcode,book_page_count,book_released_year,book_detail,clean,labels
40667,Satranç,Katip Yayınları,Stefan Zweig,literature,9786256975316,88,2022,Stefan Zweig’ın intihar etmeden kısa süre önc...,intihar etmeden kısa süre önce kaleme aldığı s...,1
41266,Satranç,Bilgili Yayınları,Stefan Zweig,literature,9786057490476,80,2022,"Satranç, du¨nyaca tanınmış Avusturya-Macarista...",satranç du¨nyaca tanınmış avusturyamacaristanl...,1
44866,Stefan Zweig 7 Kitap Set,Girdap Kitap,Stefan Zweig,literature,9786057113870,508,2022,Satranç\r\nBilinmeyen Bir Kadının Mektubu\r\nK...,satranç bilinmeyen bir kadının mektubu korku b...,1
46301,Stefan Zweig ve Sherlock Holmes Seti 16 Kitap,Maviçatı Yayınları,Stefan Zweig,literature,9915923979764,1944,NaN,Stefan Zweig Seti 10 Kitap (Set-1)\r\n- Kuş Ka...,seti 10 kitap set1 kuş kapanı dönüşüm 86 sayfa...,1
46309,Stefan Zweig 1-2 ve Franz Kafka Seti 30 Kitap,Maviçatı Yayınları,Stefan Zweig,literature,9915923979849,2112,NaN,Stefan Zweig Seti 10 Kitap (Set-1)\r\n- Kuş Ka...,seti 10 kitap set1 kuş kapanı dönüşüm 86 sayfa...,1
46330,Stefan Zweig 1-2 - Franz Kafka ve Sherlock Hol...,Maviçatı Yayınları,Stefan Zweig,literature,9915923979924,3312,NaN,Stefan Zweig Seti 10 Kitap (Set-1)\r\n- Kuş Ka...,seti 10 kitap set1 kuş kapanı dönüşüm 86 sayfa...,1
46679,Stefan Zweig Seti - 11 Kitap,Bilgi Yayınevi,Stefan Zweig,literature,9786054875429,858,2019,Satranç\rBilgi Yayınevi`nden özenle hazırlanmı...,satranç bilgi yayınevinden özenle hazırlanmış ...,1
46802,Satranç,Mavi Nefes Yayınları,Stefan Zweig,literature,9786258054446,115,2022,Eserleriyle Dünya Edebiyatına damgasını vurmuş...,eserleriyle dünya edebiyatına damgasını vurmuş...,1
51786,Bir Kadının Hayatından Yirmidört Saat,Venedik Yayınları,Stefan Zweig,literature,9786052444733,120,2021,"Stefan Zweig, 1881’de Viyana’da dünyaya geldi....",1881de viyanada dünyaya geldi avrupa da iyi bi...,1
51940,Geçmişe Yolculuk,Halk Kitabevi,Stefan Zweig,literature,9786257468985,56,2022,"Stefan Zweig, 1881`de Viyana`da dünyaya geldi....",1881de viyanada dünyaya geldi avrupada iyi bir...,1
